In [ ]:
!pip install tensorflow opencv-python kagglehub torch torchvision transformers pillow requests


In [ ]:
import kagglehub

# Download FER2013 dataset once
path = kagglehub.dataset_download("msambare/fer2013")

print("✅ FER2013 dataset path:", path)


Using Colab cache for faster access to the 'fer2013' dataset.
✅ FER2013 dataset path: /kaggle/input/fer2013


In [ ]:
import cv2
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# -----------------------------
# Paths and Constants
# -----------------------------
TRAIN_DIR = 'dataset/train'
IMG_SIZE = (48, 48)
BATCH_SIZE = 32

# -----------------------------
# Step 1: Preprocess Images using OpenCV
# -----------------------------
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_and_save_images(source_dir, target_dir):
    """
    Reads images, detects faces, converts to grayscale,
    resizes, normalizes, and saves processed images.
    """
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for emotion_folder in os.listdir(source_dir):
        emotion_path = os.path.join(source_dir, emotion_folder)
        target_folder = os.path.join(target_dir, emotion_folder)
        os.makedirs(target_folder, exist_ok=True)

        for img_name in os.listdir(emotion_path):
            img_path = os.path.join(emotion_path, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)

            for (x, y, w, h) in faces:
                face = gray[y:y+h, x:x+w]
                resized = cv2.resize(face, IMG_SIZE)
                normalized = resized / 255.0

                save_path = os.path.join(target_folder, img_name)
                cv2.imwrite(save_path, (normalized * 255))  # Save processed face image
                break  # Process only one face per image

# Example: preprocess training data
preprocess_and_save_images('raw_dataset/train', TRAIN_DIR)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.1  # 10% for validation
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    subset='validation'
)


Found 25841 images belonging to 7 classes.
Found 2868 images belonging to 7 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization

def build_emotion_model(input_shape=(48, 48, 1), num_classes=7):
    model = Sequential([
        Conv2D(64, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),

        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.3),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

emotion_model = build_emotion_model()
emotion_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │     4,719,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         3,59

 Total params: 5,870,535 (22.39 MB)

 Trainable params: 5,868,743 (22.39 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

EPOCHS = 50

early_stop = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)
checkpoint = ModelCheckpoint('best_emotion_model.h5', monitor='val_accuracy', save_best_only=True)

history = emotion_model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint]
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.2273 - loss: 2.7197

404/404 ━━━━━━━━━━━━━━━━━━━━ 83s 178ms/step - accuracy: 0.2273 - loss: 2.7180 - val_accuracy: 0.2471 - val_loss: 1.8400 - learning_rate: 5.0000e-04
Epoch 2/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.2812 - loss: 1.7594

404/404 ━━━━━━━━━━━━━━━━━━━━ 52s 128ms/step - accuracy: 0.2812 - loss: 1.7593 - val_accuracy: 0.2891 - val_loss: 1.7939 - learning_rate: 5.0000e-04
Epoch 3/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.3213 - loss: 1.6691

404/404 ━━━━━━━━━━━━━━━━━━━━ 52s 129ms/step - accuracy: 0.3213 - loss: 1.6690 - val_accuracy: 0.4248 - val_loss: 1.4943 - learning_rate: 5.0000e-04
Epoch 4/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.3752 - loss: 1.5724

...

404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 126ms/step - accuracy: 0.6548 - loss: 0.9349 - val_accuracy: 0.6429 - val_loss: 0.9813 - learning_rate: 2.5000e-04
Epoch 37/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 52s 129ms/step - accuracy: 0.6502 - loss: 0.9407 - val_accuracy: 0.6350 - val_loss: 0.9841 - learning_rate: 2.5000e-04
Epoch 38/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 50s 125ms/step - accuracy: 0.6533 - loss: 0.9355 - val_accuracy: 0.6406 - val_loss: 0.9692 - learning_rate: 2.5000e-04
Epoch 39/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.6637 - loss: 0.9187

404/404 ━━━━━━━━━━━━━━━━━━━━ 52s 128ms/step - accuracy: 0.6637 - loss: 0.9187 - val_accuracy: 0.6434 - val_loss: 0.9633 - learning_rate: 2.5000e-04
Epoch 40/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 52s 128ms/step - accuracy: 0.6627 - loss: 0.9146 - val_accuracy: 0.6362 - val_loss: 0.9962 - learning_rate: 2.5000e-04
Epoch 41/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 126ms/step - accuracy: 0.6611 - loss: 0.9062 - val_accuracy: 0.6421 - val_loss: 0.9921 - learning_rate: 2.5000e-04
Epoch 42/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 127ms/step - accuracy: 0.6593 - loss: 0.9126 - val_accuracy: 0.6404 - val_loss: 0.9962 - learning_rate: 2.5000e-04
Epoch 43/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.6687 - loss: 0.8929

404/404 ━━━━━━━━━━━━━━━━━━━━ 54s 133ms/step - accuracy: 0.6687 - loss: 0.8929 - val_accuracy: 0.6478 - val_loss: 0.9843 - learning_rate: 2.5000e-04
Epoch 44/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.6760 - loss: 0.8775

404/404 ━━━━━━━━━━━━━━━━━━━━ 54s 134ms/step - accuracy: 0.6760 - loss: 0.8775 - val_accuracy: 0.6500 - val_loss: 0.9642 - learning_rate: 1.2500e-04
Epoch 45/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.6786 - loss: 0.8753

404/404 ━━━━━━━━━━━━━━━━━━━━ 53s 131ms/step - accuracy: 0.6786 - loss: 0.8753 - val_accuracy: 0.6592 - val_loss: 0.9590 - learning_rate: 1.2500e-04
Epoch 46/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 126ms/step - accuracy: 0.6831 - loss: 0.8542 - val_accuracy: 0.6507 - val_loss: 0.9863 - learning_rate: 1.2500e-04
Epoch 47/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 125ms/step - accuracy: 0.6883 - loss: 0.8493 - val_accuracy: 0.6581 - val_loss: 0.9686 - learning_rate: 1.2500e-04
Epoch 48/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 125ms/step - accuracy: 0.6865 - loss: 0.8411 - val_accuracy: 0.6567 - val_loss: 0.9585 - learning_rate: 1.2500e-04
Epoch 49/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 53s 132ms/step - accuracy: 0.6835 - loss: 0.8587 - val_accuracy: 0.6571 - val_loss: 0.9624 - learning_rate: 1.2500e-04
Epoch 50/50
404/404 ━━━━━━━━━━━━━━━━━━━━ 51s 127ms/step - accuracy: 0.6963 - loss: 0.8344 - val_accuracy: 0.6583 - val_loss: 0.9654 - learning_rate: 1.2500e-04


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Evaluate on test set
loss, acc = emotion_model.evaluate(test_gen)
print(f"Test Accuracy: {acc*100:.2f}%")

# Predict classes
y_true = test_gen.classes
y_pred_prob = emotion_model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)
class_labels = list(test_gen.class_indices.keys())

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_labels, yticklabels=class_labels, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Emotion Model")
plt.show()

# Plot Accuracy & Loss
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()


 56/113 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.5628 - loss: 1.2158